# ⚖️ Week 6 - Part 2: Normalization, Scaling & Fixing Skewness
### Data Science & Visualization · Team 1 · 24ADI204
**Objective:** Fix extreme right skewness in financial values and scale numerical features so distance and gradient-based models treat all dimensions fairly.
---
**Interconnection Pipeline Step:** `2 of 4`  
- **Input:** `encoded_transactions.csv` (from Notebook 1: Categorical Encoding)  
- **Output:** `scaled_transformed_transactions.csv` (saved for Notebook 3: Feature Selection)


In [ ]:
# 1. Setup & Environment Imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
print('Libraries loaded!')


In [ ]:
# 2. Loading Interconnected Dataset from Notebook 1
input_file = 'encoded_transactions.csv'
if not os.path.exists(input_file):
    raise FileNotFoundError(f'{input_file} not found! Please run Notebook 1 first.')

df = pd.read_csv(input_file)
print(f'Loaded {len(df):,} records from Notebook 1. Columns: {df.columns.tolist()}')
df.head(3)


## Step 1: Measuring Skewness
Skewness measures asymmetry around the mean:
- Skewness between `-0.5` and `+0.5` is roughly symmetrical.
- Skewness `> +1.0` is **highly right-skewed** (typical for financial transactions where most are small and a few are thousands of dollars).


In [ ]:
skewness_report = df[['amt', 'city_pop', 'customer_age', 'transaction_hour']].skew()
print('Feature Skewness Coefficients:')
print(skewness_report)


## Step 2: Fixing Skewness via Log Transformation $\ln(1 + x)$
Because `amt` and `city_pop` have severe right skewness (> 10), we apply `np.log1p(x) = ln(1 + x)`.
Adding `1` prevents $\ln(0)$ errors while smoothly contracting extreme outliers.


In [ ]:
raw_amt_skew = df['amt'].skew()
df['log_amt'] = np.log1p(df['amt'])
log_amt_skew = df['log_amt'].skew()

raw_pop_skew = df['city_pop'].skew()
df['log_city_pop'] = np.log1p(df['city_pop'])
log_pop_skew = df['log_city_pop'].skew()

print(f'Amount Skewness:   Raw = {raw_amt_skew:.2f}  -->  Log-Transformed = {log_amt_skew:.2f}')
print(f'City Pop Skewness: Raw = {raw_pop_skew:.2f}  -->  Log-Transformed = {log_pop_skew:.2f}')


In [ ]:
# Visualizing Before vs After Skewness Fix
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df['amt'], bins=50, color='#e74c3c', ax=axes[0])
axes[0].set_title(f'Raw Amount (Skew = {raw_amt_skew:.2f})', fontweight='bold')
axes[0].set_xlabel('Amount ($)')

sns.histplot(df['log_amt'], bins=50, color='#2ecc71', ax=axes[1])
axes[1].set_title(f'Log-Transformed Amount (Skew = {log_amt_skew:.2f})', fontweight='bold')
axes[1].set_xlabel('log(1 + Amount)')
plt.tight_layout()
plt.show()


## Step 3: Feature Scaling (StandardScaler vs MinMaxScaler vs RobustScaler)
Why scaling is essential:
- `amt` spans \$1 to \$10,000+.
- `transaction_hour` spans 0 to 23.
- Without scaling, distance calculations (KNN, PCA, K-Means) and gradients are dominated solely by the large numbers.


In [ ]:
features_to_scale = ['log_amt', 'log_city_pop', 'customer_age', 'transaction_hour', 'category_code']

# Standard Scaler: z = (x - mean) / std (Zero mean, Unit variance)
std_scaler = StandardScaler()
df_standardized = pd.DataFrame(std_scaler.fit_transform(df[features_to_scale]), columns=[f'{c}_std' for c in features_to_scale])

# Min-Max Scaler: scales to range [0, 1]
minmax_scaler = MinMaxScaler()
df_normalized = pd.DataFrame(minmax_scaler.fit_transform(df[features_to_scale]), columns=[f'{c}_minmax' for c in features_to_scale])

print('StandardScaler Result (Mean ≈ 0, Std ≈ 1):')
print(df_standardized.describe().T[['mean', 'std', 'min', 'max']])


## Step 4: Comparing Scaler Distributions
Let us compare how StandardScaler and MinMaxScaler transform the `log_amt` feature.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.kdeplot(df_standardized['log_amt_std'], color='#3498db', fill=True, ax=axes[0])
axes[0].set_title('StandardScaler Distribution (Mean=0, Std=1)', fontweight='bold')

sns.kdeplot(df_normalized['log_amt_minmax'], color='#9b59b6', fill=True, ax=axes[1])
axes[1].set_title('MinMaxScaler Distribution (Range: 0 to 1)', fontweight='bold')
plt.tight_layout()
plt.show()


## Step 5: Saving Scaled Dataset for Notebook 3
We save the standardized numeric features alongside encoded categories and the target column.


In [ ]:
# Add Haversine Distance feature
def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dp, dl = np.radians(lat2 - lat1), np.radians(lon2 - lon1)
    a = np.sin(dp/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2 * r * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

df['distance_km'] = haversine_km(df['lat'], df['long'], df['merch_lat'], df['merch_long'])
df['is_night'] = ((df['transaction_hour'] >= 22) | (df['transaction_hour'] <= 3)).astype(int)

# Final scaled feature set
scale_cols = ['log_amt', 'log_city_pop', 'customer_age', 'transaction_hour', 'distance_km', 'category_code']
scaled_array = std_scaler.fit_transform(df[scale_cols])
scaled_df = pd.DataFrame(scaled_array, columns=scale_cols)

# Keep unscaled binary flags
scaled_df['gender_encoded'] = df['gender_encoded'].values
scaled_df['is_night'] = df['is_night'].values
scaled_df['state_freq'] = df['state_freq'].values
scaled_df['is_fraud'] = df['is_fraud'].values

output_file = 'scaled_transformed_transactions.csv'
scaled_df.to_csv(output_file, index=False)
print(f'Successfully saved scaled dataset to: {output_file} (Shape: {scaled_df.shape})')
